# 01 — Exploration

**Objectif du projet** : construire un modèle de NLP capable de prédire automatiquement la catégorie d'un article de presse à partir de son texte, puis le rendre utilisable par un tiers via une API et une interface web.

**Dataset** : *AG News* (Hugging Face, `fancyzhx/ag_news`) : plus de 120 000 articles de presse en anglais, collectés en 2004-2005 par le moteur de recherche d'actualités ComeToMyHead, et répartis en 4 catégories : **World, Sports, Business, Sci/Tech**.

**Objectif de ce notebook** : comprendre les données avant de les nettoyer : structure, qualité, répartition des catégories, longueur des textes et bruit présent.

## 0. Installation et imports

On installe la bibliothèque `datasets` de Hugging Face, qui permet de télécharger le dataset directement depuis le Hub. On importe ensuite :
- `pandas` / `numpy` : manipulation des données
- `matplotlib` / `seaborn` : visualisations
- `re` : expressions régulières pour rechercher des motifs dans les textes

`%pip` (plutôt que `!pip`) installe le paquet dans l'environnement du notebook en cours : cela fonctionne aussi bien sur Colab qu'en local. Le filtre `warnings` masque un avertissement de pandas sans conséquence, lié aux expressions régulières utilisées plus bas.

In [ ]:
%pip install -q datasets

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

## 1. Chargement des données

Le dataset est déjà découpé en un jeu d'entraînement (`train`) et un jeu de test (`test`). Les catégories sont stockées sous forme de numéros (0 à 3) ; leurs noms sont dans les métadonnées du dataset. On convertit les deux jeux en DataFrames pandas et on ajoute une colonne `label_text` avec le nom de la catégorie.

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
dataset

In [ ]:
# Noms des catégories stockés dans les métadonnées du dataset
noms_categories = dataset["train"].features["label"].names
noms_categories

In [ ]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

for df in (train_df, test_df):
    df["label_text"] = df["label"].map(dict(enumerate(noms_categories)))

print("Train :", train_df.shape)
print("Test  :", test_df.shape)
print("Total :", len(train_df) + len(test_df), "articles")

In [ ]:
train_df.head()

**Observations** :
- Le dataset contient **127 600 articles** : **120 000** en train et **7 600** en test. C'est largement au-dessus du seuil d'environ 1 000 observations recommandé pour limiter le surapprentissage.
- **4 catégories** : 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech.
- Chaque texte est composé du **titre** de l'article suivi de sa **description** (le chapeau), sans séparateur.
- Le texte n'est pas nettoyé : majuscules, ponctuation, mention de la source (*(Reuters) Reuters -*) et caractères parasites (`\` au milieu des phrases).

## 2. Exploration des données

### 2.1 Structure et types

On vérifie le type de chaque colonne et la mémoire utilisée.

In [ ]:
train_df.info()

**Observations** : 3 colonnes au bon type (deux colonnes texte et `label` en entier), aucune valeur nulle signalée. Le jeu d'entraînement occupe environ **31 Mo** en mémoire.

### 2.2 Valeurs manquantes et textes vides

Un texte manquant ou vide ne peut pas être classé : on vérifie qu'il n'y en a pas.

In [ ]:
for nom, df in (("train", train_df), ("test", test_df)):
    print(f"{nom} : valeurs manquantes = {df.isnull().sum().sum()}, textes vides = {(df['text'].str.strip() == '').sum()}")

**Observations** : aucune valeur manquante et aucun texte vide dans les deux jeux.

### 2.3 Doublons

On cherche trois types de doublons :
1. les **doublons exacts** (même texte au caractère près) ;
2. les **quasi-doublons** : le même article à une différence de ponctuation, d'espaces ou de majuscules près. Pour les détecter, on crée une **clé** qui ne garde que les lettres en minuscules ;
3. les **articles en double avec des catégories différentes**, qui envoient des signaux contradictoires au modèle.

In [ ]:
print("Doublons exacts — train :", train_df.duplicated(subset="text").sum())
print("Doublons exacts — test  :", test_df.duplicated(subset="text").sum())
print("Articles identiques entre train et test :", len(set(train_df["text"]) & set(test_df["text"])))

In [ ]:
# Quasi-doublons : on compare les textes en ne gardant que les lettres en minuscules
def normaliser(texte):
    return re.sub(r"[^a-z]", "", texte.lower())

train_df["cle"] = train_df["text"].apply(normaliser)
test_df["cle"] = test_df["text"].apply(normaliser)

print("Quasi-doublons — train :", train_df.duplicated(subset="cle").sum())
print("Quasi-doublons — test  :", test_df.duplicated(subset="cle").sum())
print("Quasi-doublons entre train et test :", len(set(train_df["cle"]) & set(test_df["cle"])))

In [ ]:
# Un même article classé dans plusieurs catégories différentes ?
groupes = train_df[train_df.duplicated(subset="cle", keep=False)].groupby("cle")["label_text"]
contradictoires = groupes.nunique()
print("Articles en double avec des catégories différentes :", (contradictoires > 1).sum())

exemple = contradictoires[contradictoires > 1].index[0]
train_df.loc[train_df["cle"] == exemple, ["text", "label_text"]]

**Observations** :
- **Aucun doublon exact**, mais **424 quasi-doublons** dans le train, 1 dans le test et **38 articles communs** au train et au test, ce qui représente une **fuite de données** : le modèle serait évalué sur des articles qu'il a déjà vus.
- **44 articles** apparaissent dans le train avec **des catégories différentes**. Exemple : l'article sur le rachat de Joyo.com par Amazon est classé à la fois en *Business* et en *Sci/Tech*. Ces étiquettes contradictoires sont du bruit pour le modèle.

➡️ Ces trois problèmes seront traités dans le notebook `02_preprocessing`.

### 2.4 Répartition des catégories

On vérifie si les classes sont équilibrées : un fort déséquilibre pousserait le modèle à favoriser la classe majoritaire et rendrait l'accuracy trompeuse.

In [ ]:
repartition = pd.DataFrame({
    "train": train_df["label_text"].value_counts(),
    "test": test_df["label_text"].value_counts(),
})
repartition["train_%"] = (repartition["train"] / repartition["train"].sum() * 100).round(1)
repartition["test_%"] = (repartition["test"] / repartition["test"].sum() * 100).round(1)
repartition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ordre = noms_categories

sns.countplot(data=train_df, x="label_text", order=ordre, ax=axes[0], color="steelblue")
axes[0].set_title("Répartition des catégories — train")
axes[0].set_xlabel("Catégorie")
axes[0].set_ylabel("Nombre d'articles")

sns.countplot(data=test_df, x="label_text", order=ordre, ax=axes[1], color="darkorange")
axes[1].set_title("Répartition des catégories — test")
axes[1].set_xlabel("Catégorie")
axes[1].set_ylabel("Nombre d'articles")

plt.tight_layout()
plt.show()

**Observations** : le dataset est **parfaitement équilibré** : exactement **30 000 articles par catégorie** en train (25 %) et **1 900** en test. Aucun rééquilibrage n'est nécessaire, et l'accuracy sera une métrique fiable (on suivra aussi le F1-score macro).

### 2.5 Longueur des articles

On calcule le nombre de caractères et de mots de chaque article. La longueur influence certains choix de modélisation, par exemple la longueur maximale des séquences pour le LSTM et DistilBERT.

In [ ]:
for df in (train_df, test_df):
    df["nb_caracteres"] = df["text"].str.len()
    df["nb_mots"] = df["text"].str.split().str.len()

train_df[["nb_caracteres", "nb_mots"]].describe().round(1)

In [ ]:
train_df.groupby("label_text")["nb_mots"].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(train_df["nb_mots"], bins=60, ax=axes[0], color="steelblue")
axes[0].axvline(train_df["nb_mots"].median(), color="red", linestyle="--", label="médiane")
axes[0].set_title("Distribution du nombre de mots par article (train)")
axes[0].set_xlabel("Nombre de mots")
axes[0].set_ylabel("Nombre d'articles")
axes[0].legend()

sns.boxplot(data=train_df, x="label_text", y="nb_mots", order=ordre, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Nombre de mots par catégorie (train)")
axes[1].set_xlabel("Catégorie")
axes[1].set_ylabel("Nombre de mots")

plt.tight_layout()
plt.show()

**Observations** :
- Les textes sont **courts** : **37 mots** en médiane (236 caractères en moyenne) et la moitié des articles font entre 32 et 43 mots. C'est logique, puisqu'on n'a que le titre et le chapeau, pas l'article complet.
- La distribution est concentrée, avec quelques textes plus longs (jusqu'à **177 mots**).
- Les 4 catégories ont des longueurs **très proches** (médianes de 36 à 39 mots) : la longueur ne permet pas de distinguer les catégories, seul le vocabulaire le pourra. *Sci/Tech* est la plus variable, avec le plus de textes longs.
- Pour DistilBERT, une longueur maximale de **128 tokens** suffira à couvrir la quasi-totalité des textes sans les tronquer.

### 2.6 Exemples d'articles par catégorie

On affiche deux articles de chaque catégorie pour se faire une idée du contenu et du vocabulaire.

In [ ]:
for categorie in ordre:
    print(f"=== {categorie.upper()} ===")
    for texte in train_df.loc[train_df["label_text"] == categorie, "text"].iloc[:2]:
        print("-", texte[:250])
    print()

**Observations** :
- Le vocabulaire est très marqué : *referendum, president, police, troops* (World), *Olympic, freestyle, homered, wild-card* (Sports), *Wall Street, investment, stock market* (Business).
- On retrouve dans presque tous les textes la mention de la **source** (*(Reuters) Reuters -*, *(AP) AP -*) et le caractère `\` au milieu des phrases, qui remplace un saut de ligne.

### 2.7 Bruit dans les textes

Les articles ont été collectés automatiquement sur des sites web : ils contiennent des restes de code HTML. On compte le nombre d'articles concernés par chaque type de bruit.

In [ ]:
motifs = {
    "entité #39; (apostrophe)": r"#39;",
    "entité #36; (dollar)": r"#36;",
    "entité quot; (guillemet)": r"quot;",
    "balise HTML (&lt;...&gt;)": r"&lt;",
    "barre oblique inverse \\": r"\\",
    "lien http": r"http",
    "mention de source (Reuters) Reuters -": r"\(([^()]{1,40})\)\s*\1\s*-",
}

bruit = pd.DataFrame({
    "nb_articles": {nom: train_df["text"].str.contains(m, regex=True).sum() for nom, m in motifs.items()}
})
bruit["%_du_train"] = (bruit["nb_articles"] / len(train_df) * 100).round(2)
bruit.sort_values("nb_articles", ascending=False)

In [ ]:
# Exemple d'article contenant de la publicité HTML
print(train_df.loc[train_df["text"].str.contains("ADVERTISEMENT"), "text"].iloc[0][:600])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
b = bruit.sort_values("nb_articles")
ax.barh(b.index, b["nb_articles"], color="indianred")
ax.set_title("Nombre d'articles du train contenant chaque type de bruit")
ax.set_xlabel("Nombre d'articles")
plt.tight_layout()
plt.show()

**Observations** :
- **24,8 % des articles** contiennent l'entité **`#39;`**, un code HTML mal converti qui remplace l'apostrophe (*Google #39;s* au lieu de *Google's*). De même, **`#36;`** remplace le symbole dollar et **`quot;`** les guillemets.
- **13,4 %** commencent par une **mention de source** (*(Reuters) Reuters -*) et **11 %** contiennent des **`\`** à la place de sauts de ligne.
- **4,4 %** contiennent des **balises HTML** encodées (`&lt;p&gt;`, `&lt;a href=...&gt;`), parfois des **blocs publicitaires** entiers avec des liens (*ADVERTISEMENT… Introducing IBM eServer p5 systems*).

➡️ Ce bruit n'a rien à voir avec le sujet de l'article : il devra être supprimé avant l'entraînement.

### 2.8 Sources des articles par catégorie

On extrait la source mentionnée en début d'article (Reuters, AP, AFP…) et on regarde comment les articles de chaque source se répartissent entre les catégories.

In [ ]:
train_df["source"] = train_df["text"].str.extract(r"\(([^()]{1,40})\)\s*\1\s*-")[0]
top_sources = train_df["source"].value_counts().head(8).index

tableau = pd.crosstab(train_df.loc[train_df["source"].isin(top_sources), "source"],
                      train_df.loc[train_df["source"].isin(top_sources), "label_text"])
tableau

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
part = tableau.div(tableau.sum(axis=1), axis=0) * 100
sns.heatmap(part, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% des articles de la source"}, ax=ax)
ax.set_title("Répartition des articles de chaque source par catégorie (%)")
ax.set_xlabel("Catégorie")
ax.set_ylabel("Source")
plt.tight_layout()
plt.show()

**Observations** :
- Certaines sources sont **liées à une seule catégorie** : *Canadian Press* → 100 % World, *PC World* et *SPACE.com* → 100 % Sci/Tech, *NewsFactor* → 98 % Sci/Tech, *AFP* → 77 % World.
- Un modèle pourrait donc apprendre à classer un article **d'après sa source** plutôt que d'après son contenu. Ce raccourci fonctionnerait sur ce dataset, mais pas sur un texte saisi par un utilisateur dans l'application, qui ne contient pas de mention de source.

➡️ On **supprimera les mentions de source** pour que le modèle apprenne à partir du contenu de l'article.

## 3. Bilan de l'exploration

| Constat | Action dans `02_preprocessing` |
|---|---|
| 127 600 articles, 4 catégories parfaitement équilibrées | Aucun rééquilibrage nécessaire |
| Textes courts (37 mots en médiane) | Longueur max de 128 tokens pour DistilBERT |
| Entités HTML (`#39;`, `quot;`, `&lt;`), `\`, liens, publicités | Fonction de suppression du bruit |
| Mentions de source fortement liées à certaines catégories | Suppression des mentions de source |
| 424 quasi-doublons, 44 articles aux catégories contradictoires | Suppression |
| 38 articles communs au train et au test (fuite de données) | Suppression du test |